# Hybrid Quantum-Classical Network Assignment — Test Notebook

**시나리오**: 6 UE × 3 AP, AP capacity = 2, 경계 UE: {2, 4}

| Cell | AP | UE (로컬) | 경계 UE |
|------|----|-----------|---------|
| A | AP0 | UE0, UE1, UE2 | UE2 |
| B | AP1 | UE2, UE3, UE4 | UE2, UE4 |
| C | AP2 | UE4, UE5 | UE4 |

In [ ]:
import numpy as np
from qiskit.visualization import circuit_drawer
import matplotlib.pyplot as plt

# hybrid_assignment_example.py에서 모듈 임포트
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from hybrid_assignment_example import (
    LINK_QUALITY, N_UE, N_AP, AP_CAPACITY, BOUNDARY_UES,
    decompose_network, build_local_grover, solve_local,
    exchange_boundary_info, reconcile_conflicts,
    centralized_optimal, compute_hybrid_score,
)

print('Import 완료')

## Step 1: Network Decomposition (네트워크 분할)

In [ ]:
subproblems = decompose_network()

print('링크 품질 행렬 (UE × AP):')
print(LINK_QUALITY)
print()

for sp in subproblems:
    print(f"{sp['name']}: UE{sp['ue_indices']} → AP{sp['ap_index']}")
    print(f"  로컬 채널 품질: {sp['qualities']}")
    print(f"  AP 용량 제한: {sp['capacity']}")

## Step 2: Local Grover 회로 구성 및 시각화

각 셀별 Grover 회로를 `build_local_grover`로 구성하고 회로 다이어그램을 확인합니다.

In [ ]:
# Cell A 회로 (3 UE, iteration=1로 구조 확인)
qc_a = build_local_grover(subproblems[0], iterations=1)
print(f"Cell A: {qc_a.num_qubits} qubits, depth = {qc_a.depth()}")
qc_a.draw('mpl', fold=80, style='iqp')

In [ ]:
# Cell B 회로 (3 UE)
qc_b = build_local_grover(subproblems[1], iterations=1)
print(f"Cell B: {qc_b.num_qubits} qubits, depth = {qc_b.depth()}")
qc_b.draw('mpl', fold=80, style='iqp')

In [ ]:
# Cell C 회로 (2 UE)
qc_c = build_local_grover(subproblems[2], iterations=1)
print(f"Cell C: {qc_c.num_qubits} qubits, depth = {qc_c.depth()}")
qc_c.draw('mpl', fold=80, style='iqp')

### 회로 비교 (3개 셀 나란히)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 16))

for idx, (sp, ax) in enumerate(zip(subproblems, axes)):
    qc = build_local_grover(sp, iterations=1)
    qc.draw('mpl', fold=60, ax=ax, style='iqp')
    ax.set_title(f"{sp['name']} — {len(sp['ue_indices'])} UE, {qc.num_qubits} qubits, depth={qc.depth()}",
                 fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 2b: Statevector 분석 (측정 전)

측정 없이 Statevector를 확인하여 oracle이 올바르게 위상 마킹하는지 검증합니다.

In [ ]:
from qiskit.quantum_info import Statevector

for sp in subproblems:
    n_ue = len(sp['ue_indices'])
    # 측정 없는 회로로 statevector 추출
    qc = build_local_grover(sp, iterations=2)
    qc_no_meas = qc.remove_final_measurements(inplace=False)
    sv = Statevector.from_instruction(qc_no_meas)
    probs = sv.probabilities_dict()

    # assign 큐비트만의 확률 추출 (flag 큐비트 무시)
    assign_probs = {}
    for bitstr, p in probs.items():
        # bitstr: flag + assign (Qiskit big-endian), assign = 마지막 n_ue 비트
        assign_bits = bitstr[-n_ue:]
        assign_probs[assign_bits] = assign_probs.get(assign_bits, 0) + p

    # 확률 내림차순 정렬
    sorted_probs = sorted(assign_probs.items(), key=lambda x: -x[1])

    print(f"\n{sp['name']} (UE{sp['ue_indices']} → AP{sp['ap_index']}):")
    print(f"  qualities = {sp['qualities']}")
    print(f"  Top states (assign 확률):")
    for bitstr, p in sorted_probs[:8]:
        # 할당 해석
        assigns = [int(b) for b in reversed(bitstr)]
        n_assigned = sum(assigns)
        feasible = '✓' if n_assigned <= sp['capacity'] else '✗'
        ue_str = ', '.join(f"UE{sp['ue_indices'][i]}" for i, a in enumerate(assigns) if a)
        print(f"    |{bitstr}⟩  P={p:.4f}  assigned={n_assigned} {feasible}  [{ue_str}]")

## Step 2c: Shot-based 실행 결과

In [ ]:
local_results = []

for sp in subproblems:
    res = solve_local(sp, shots=4096, iterations=2)
    local_results.append(res)

    ue_names = [f"UE{u}" for u in sp['ue_indices']]
    assign_str = ', '.join(
        f"{name}={'O' if a else 'X'}"
        for name, a in zip(ue_names, res['assignment'])
    )
    print(f"{sp['name']}: {assign_str}")
    sorted_counts = sorted(res['counts'].items(), key=lambda x: -x[1])[:8]
    print(f"  Top counts: {dict(sorted_counts)}")

In [ ]:
# 히스토그램 시각화
from qiskit.visualization import plot_histogram

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, res in enumerate(local_results):
    sp = res['subproblem']
    plot_histogram(res['counts'], ax=axes[idx], title=sp['name'],
                   bar_labels=False, sort='value_desc')

plt.tight_layout()
plt.show()

## Step 3: Classical Communication (경계 정보 교환)

In [ ]:
global_assignment = exchange_boundary_info(local_results)

for ue in sorted(global_assignment.keys()):
    entries = global_assignment[ue]
    marker = ' ← BOUNDARY' if ue in BOUNDARY_UES else ''
    info = ', '.join(
        f"AP{e['ap']}({'할당' if e['assigned'] else '미할당'}, q={e['quality']:.1f})"
        for e in entries
    )
    print(f"UE{ue}: {info}{marker}")

## Step 4: Reconciliation (충돌 해소)

In [ ]:
final, conflicts, ap_load = reconcile_conflicts(global_assignment)

if conflicts:
    print('충돌 감지:')
    for ue, ctype, detail in conflicts:
        print(f"  UE{ue} — {ctype} (detail={detail})")
else:
    print('충돌 없음')

print('\n최종 할당:')
for ue in sorted(final.keys()):
    ap = final[ue]
    if ap is not None:
        q = LINK_QUALITY[ue, ap]
        print(f"  UE{ue} → AP{ap} (quality={q:.1f})")
    else:
        print(f"  UE{ue} → 미할당")

print(f'\nAP 부하: {dict(ap_load)}')

## Step 5: Benchmark (중앙집중 vs 하이브리드)

In [ ]:
hybrid_score = compute_hybrid_score(final)
opt_assign, opt_score = centralized_optimal()

print(f'중앙집중 최적해: {opt_assign}')
print(f'  최적 총 품질 = {opt_score:.1f}')
print(f'하이브리드 총 품질 = {hybrid_score:.1f}')
if opt_score > 0:
    ratio = hybrid_score / opt_score * 100
    print(f'성능비: {ratio:.1f}%')

# 시각 비교
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Centralized\n(Optimal)', 'Hybrid\n(Quantum+Classical)'],
              [opt_score, hybrid_score],
              color=['#4C72B0', '#DD8452'], edgecolor='black')
ax.set_ylabel('Total Link Quality')
ax.set_title('Centralized vs Hybrid Performance')
for bar, val in zip(bars, [opt_score, hybrid_score]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Grover iteration 수에 따른 성능 변화

In [ ]:
# iteration 1~4로 반복 실험
iter_scores = []

for n_iter in range(1, 5):
    results = [solve_local(sp, shots=4096, iterations=n_iter) for sp in subproblems]
    ga = exchange_boundary_info(results)
    fa, _, _ = reconcile_conflicts(ga)
    score = compute_hybrid_score(fa)
    iter_scores.append(score)
    print(f'iterations={n_iter}: hybrid_score={score:.1f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, 5), iter_scores, 'o-', color='#4C72B0', linewidth=2)
ax.axhline(y=opt_score, color='red', linestyle='--', label=f'Optimal ({opt_score:.1f})')
ax.set_xlabel('Grover Iterations')
ax.set_ylabel('Hybrid Score')
ax.set_title('Hybrid Score vs Grover Iterations')
ax.legend()
ax.set_xticks(range(1, 5))
plt.tight_layout()
plt.show()

## 회로 상세 — Transpile 후 게이트 카운트

In [ ]:
from qiskit import transpile
from qiskit_aer import Aer

backend = Aer.get_backend('aer_simulator')

for sp in subproblems:
    qc = build_local_grover(sp, iterations=2)
    qc_t = transpile(qc, backend)
    gate_counts = dict(qc_t.count_ops())
    print(f"{sp['name']}:")
    print(f"  원본: {qc.num_qubits} qubits, depth={qc.depth()}")
    print(f"  트랜스파일: depth={qc_t.depth()}, gates={gate_counts}")